# test_2_v2 — `test_2.ipynb` 수정본

원본 `C:\t_project_1_1\test_2.ipynb`은 원본 CSV 전체(260,601행)를 읽었습니다. 여기에는 팀 `final_holdout` 52,121행의 정답도 포함됩니다.
팀 규칙상 홀드아웃의 입력-정답 관계는 탐색하지 않으므로, 이 복사본은 **팀 development 208,480행만** 읽습니다.

분석 코드는 원본과 같고, 바뀐 것은 데이터 경로와 저장 경로뿐입니다. 그래서 건수와 비율 숫자는 원본과 다르게 나옵니다.

In [1]:
# [v2] 팀 development 데이터만 사용 (final_holdout 제외)
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "preprocess.py").exists())
DEV_CSV = REPO / "data" / "development.csv"   # 원본 CSV에서 팀 홀드아웃 52,121행을 뺀 208,480행
OUT_DIR = REPO / "outputs" / "t_project_v2" / "test_2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("데이터:", DEV_CSV)
print("저장 폴더:", OUT_DIR)

데이터: C:\Users\jhj31\project_1\earthquake\data\development.csv
저장 폴더: C:\Users\jhj31\project_1\earthquake\outputs\t_project_v2\test_2


In [ ]:
"""
대/중/소지역(geo_level_1~3)을 하나의 지역 변수로 결합하고,
그 변수와 나머지 모든 컬럼의 1:1 연관성(검정통계량, p-value, 효과크기)을 계산한다.
"""
import numpy as np
import pandas as pd
from scipy import stats

CSV_PATH = DEV_CSV  # [v2] 팀 development
GEO_LEVEL = 3          # 1=대지역, 2=대+중, 3=대+중+소
MIN_COUNT = 1          # 표본이 n개 미만인 지역은 제외 (예: 30)

df = pd.read_csv(CSV_PATH).copy()

# ── 1. 지역 변수 결합 ────────────────────────────────────────────────
cols = [f"geo_level_{i}_id" for i in range(1, GEO_LEVEL + 1)]
df["geo_id"] = df[cols].astype(str).agg("_".join, axis=1)   # 예: "6_487_12198"
df["geo_code"] = pd.factorize(df["geo_id"])[0]              # 모델 입력용 정수 코드

if MIN_COUNT > 1:
    keep = df["geo_id"].value_counts()
    df = df[df["geo_id"].isin(keep[keep >= MIN_COUNT].index)]

# ── 2. 연관성 지표 ──────────────────────────────────────────────────
def anova_eta2(y, g):
    """수치형: 일원분산분석 F, p, eta^2(상관비)"""
    m, n = y.mean(), len(y)
    t = y.groupby(g).agg(["count", "mean"])
    k = len(t)
    ssb = (t["count"] * (t["mean"] - m) ** 2).sum()
    sst = ((y - m) ** 2).sum()
    F = (ssb / (k - 1)) / ((sst - ssb) / (n - k))
    return F, stats.f.sf(F, k - 1, n - k), ssb / sst

def chi2_cramers_v(x, g):
    """범주형: 카이제곱, p, Cramer's V"""
    ct = pd.crosstab(g, x)
    chi2, p, _, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    return chi2, p, v

# ── 3. geo_id vs 각 컬럼 1:1 검정 ───────────────────────────────────
skip = {"building_id", "geo_id", "geo_code", *[f"geo_level_{i}_id" for i in (1, 2, 3)]}
geo, rows = df["geo_id"], []

for c in [c for c in df.columns if c not in skip]:
    s = df[c]
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 10:
        stat, p, eff = anova_eta2(s, geo)
        rows.append([c, "수치형", "ANOVA F", stat, p, "eta2", eff])
    else:
        stat, p, eff = chi2_cramers_v(s, geo)
        rows.append([c, "범주형", "chi2", stat, p, "Cramers V", eff])

result = (pd.DataFrame(rows, columns=["column", "dtype", "test", "stat", "p_value",
                                      "effect", "effect_size"])
          .sort_values("effect_size", ascending=False)
          .reset_index(drop=True))

print(f"지역 수: {df['geo_id'].nunique():,} / 행 수: {len(df):,}")
print(result.to_string())
result.to_csv(OUT_DIR / "geo_association.csv", index=False)  # [v2]

지역 수: 11,353 / 행 수: 208,480
                                    column dtype     test           stat   p_value     effect  effect_size
0            has_superstructure_stone_flag   범주형     chi2  160165.074016  0.000000  Cramers V     0.876500
1                has_superstructure_timber   범주형     chi2  158313.534009  0.000000  Cramers V     0.871419
2                has_superstructure_bamboo   범주형     chi2  144661.189358  0.000000  Cramers V     0.832998
3      has_superstructure_mud_mortar_stone   범주형     chi2  139897.919458  0.000000  Cramers V     0.819169
4             has_superstructure_adobe_mud   범주형     chi2  137581.982823  0.000000  Cramers V     0.812360
5      has_superstructure_mud_mortar_brick   범주형     chi2  118984.369032  0.000000  Cramers V     0.755462
6                 has_superstructure_other   범주형     chi2  109834.945096  0.000000  Cramers V     0.725835
7                   land_surface_condition   범주형     chi2  210435.277595  0.000000  Cramers V     0.710415
8        